# Synthetic vs. Real — Google Colab GPU training

This notebook trains five YOLOv8n models with **300 training images each**:
real only, AI only, rendered only, 50% AI + 25% rendered + 25% real, and
50% real + 25% AI + 25% rendered. All use the same **7 real validation images
and 22 real test images**. Classes: `0 worker`, `1 helmet`, `2 vest`.

**Existing labels are used without further annotation review**, as requested.
Previously rejected images and missing files are excluded. Some selected
training labels remain unresolved in the old review ledger; structural checks
do not certify their accuracy. The full original v4 review remains incomplete.
Holdouts are small and contain correlated frames, so results are preliminary.
Do not compare these scores directly with the bundled v2 model scores.

**Before running:**
1. Upload `synthreal_300.zip` into a `SynthReal` folder in **My Drive**.
2. In Colab choose **Runtime → Change runtime type → GPU** (T4 if available).
3. Run the cells in order. Allow the Google Drive connection when prompted.

Your computer only needs a browser and internet. Training runs on Colab's GPU.
Colab GPU availability and runtime limits vary; checkpoints are saved to Drive.
[Colab FAQ](https://research.google.com/colaboratory/faq.html) ·
[Ultralytics training and resume documentation](https://docs.ultralytics.com/modes/train/)


## 1. Check the GPU


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


## 2. Connect Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Copy and verify the dataset
If your ZIP is somewhere else, change `DRIVE_ARCHIVE` below. Images are copied
onto the Colab machine for training speed; checkpoints stay on Drive.
The archive checksum must match this notebook version.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import shutil
import stat
import tempfile
import zipfile

DRIVE_ARCHIVE = Path('/content/drive/MyDrive/SynthReal/synthreal_300.zip')
EXPECTED_SHA256 = '5ea8644bd890bd7ab2ce79b17a20c9fe4ce43baf10fd6541b2b9ceb24bae58cf'
DATA_ROOT = Path('/content/synthreal_' + EXPECTED_SHA256[:12])
LOCAL_ARCHIVE = Path('/content/synthreal_300.zip')

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not DRIVE_ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload the supplied ZIP to {DRIVE_ARCHIVE}, or update DRIVE_ARCHIVE.')
if not LOCAL_ARCHIVE.exists() or file_sha256(LOCAL_ARCHIVE) != EXPECTED_SHA256:
    shutil.copyfile(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
if file_sha256(LOCAL_ARCHIVE) != EXPECTED_SHA256:
    raise ValueError('ZIP checksum mismatch. Use the archive supplied with this notebook.')
if not DATA_ROOT.exists():
    temporary = Path(tempfile.mkdtemp(prefix='synthreal_extract_', dir='/content'))
    with zipfile.ZipFile(LOCAL_ARCHIVE) as bundle:
        for member in bundle.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename:
                raise ValueError('Unsafe ZIP member: ' + member.filename)
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('ZIP symlinks are not supported')
        bundle.extractall(temporary)
    temporary.replace(DATA_ROOT)
print('Dataset:', DATA_ROOT)


## 4. Install the training library


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', str(DATA_ROOT / 'requirements-colab.txt')])
# Keep the CUDA-enabled torch installation supplied by Colab.
sys.path.insert(0, str(DATA_ROOT))
from colab_training import verify_and_prepare, run_training


## 5. Validate every selected image, label, split, and checksum
This step creates paths for the current Colab machine. It rejects changed
annotations, exact duplicate images, overlapping holdouts, and known source
scene groups crossing splits. It does not certify that every original source
scene is independent; source video IDs are unavailable.


In [ ]:
prepared = verify_and_prepare(DATA_ROOT)
if prepared['manifest']['train_images_per_experiment'] != 300:
    raise ValueError('This notebook requires the 300-image comparison bundle.')
import pandas as pd
table = []
for experiment in prepared['experiments']:
    table.append({'experiment': experiment['name'],
                  **experiment['sources'],
                  'train': len(experiment['splits']['train']),
                  'val': len(experiment['splits']['val']),
                  'test': len(experiment['splits']['test'])})
display(pd.DataFrame(table).fillna(0))
print('All selected files and split checks passed. Annotation accuracy has not been certified.')


## 6. Train or resume all five experiments
The defaults use 50 epochs, 640-pixel input, batch 8, seed 42, AdamW, and GPU 0.
Each experiment starts from the same YOLOv8n pretrained weights. The first
training call downloads those weights. Model selection uses validation only;
the test set is evaluated after training.

After a disconnect, reconnect to a GPU and rerun the notebook. Completed
experiments are skipped and incomplete ones resume from `weights/last.pt`.
At least one saved epoch is needed to resume. Periodic epoch checkpoints are
also kept. Do not change data or settings while resuming the same run folder.
If you deliberately change settings, give `RUN_NAME` a new value. If GPU memory
is insufficient, use a smaller batch in a **new** run folder for all five models.


In [ ]:
EPOCHS = 50
BATCH = 8
RUN_NAME = 'existing_labels_300_seed42_' + EXPECTED_SHA256[:8]
OUTPUT = Path('/content/drive/MyDrive/SynthReal/runs') / RUN_NAME
results = run_training(prepared, OUTPUT, epochs=EPOCHS, batch=BATCH)
print('Saved results and checkpoints:', OUTPUT)


## 7. Read the comparison


In [ ]:
comparison = pd.read_csv(OUTPUT / 'comparison_metrics.csv')
display(comparison)
print('Each experiment folder contains best.pt, last.pt, plots, and test_metrics.json.')
print('Small, curated holdouts: treat these metrics as preliminary.')


## 8. Export the completed models and metrics
The ZIP stays in Google Drive. It includes all five best checkpoints, the
dataset manifest, settings, and measured metrics. Full resumable runs remain
in the run folder; this smaller export omits optimizer checkpoints.


In [ ]:
export_path = OUTPUT.parent / (RUN_NAME + '_models_and_metrics.zip')
paths = [OUTPUT / name for name in ['comparison_metrics.csv', 'run_specification.json',
         'dataset_manifest.json', 'environment.json', 'initial_weights.json', 'status.json']]
for experiment in prepared['experiments']:
    run = OUTPUT / experiment['name']
    paths.extend([run / 'weights/best.pt', run / 'test_metrics.json'])
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise RuntimeError('Complete training before export. Missing: ' + ', '.join(missing))
temporary_export = export_path.with_suffix('.tmp')
with zipfile.ZipFile(temporary_export, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in paths:
        bundle.write(path, path.relative_to(OUTPUT).as_posix())
temporary_export.replace(export_path)
print('Download this file from Google Drive:', export_path)
